In [57]:
# importing requests for making HTTP requests
import requests

# importing pandas for data manipulation
import pandas as pd

# importing BeautifulSoup for parsing HTML content
from bs4 import BeautifulSoup

# a list to hold the 
books_data = []

# setting headers to mimic a browser request
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-GB,en;q=0.9',
    'Cookie': 'abe_prefs=%7B%22modified%22%3Afalse%2C%22curr%22%3A%22GBP%22%2C%22shipdest%22%3A%22GBR%22%7D; abeconsent=%221%3A1%2C2%3A0%2C3%3A0%2C4%3A0%22; abe_vc=98'
}

# creating a session to persist certain parameters across requests
s = requests.Session()
s.headers.update(headers)

# initialising page number and item count for pagination
page_number = 0
items = 0

# looping through pages until we reach the limit (10 pages for testing)
while page_number < 2:  # limit to first 2 pages for testing, ~40 books
    
    # using try-except to handle any potential errors during the scraping process
    try:
        # printing the current page number for tracking progress
        print("page: ", page_number)

        # constructing the URL for the current page of search results
        url = f'https://www.abebooks.co.uk/servlet/SearchResults?ds=20&dym=on&kn=data%20engineering&p={page_number}&rollup=on&sortby=20&sp=0&spo={items}'
        
        # printing the URL being accessed for debugging purposes
        print(url)
        
        # making a GET request to the URL and storing the response
        r = s.get(url)

        # parsing the HTML content of the response using BeautifulSoup
        soup = BeautifulSoup(r.content, 'html.parser')

        # finding all the books in the page (~20 per page)
        books = soup.find_all("li", itemtype="http://schema.org/Book")

        # looping through each book found and extracting relevant information
        for book in books:

            price_tag = book.find("meta", itemprop="price")
            # price_tag = book.find("p", {"class": "item-price"})
            # price_value = price_tag.get_text(strip=True) if price_tag else None
            currency_meta = book.find("meta", itemprop="priceCurrency")

            # creating a dictionary to hold the extracted information for each book
            book_dict = {
                "Listing ID": book.get("data-csa-c-item-id"),
                "Title": book.find("meta", itemprop="name")["content"] if book.find("meta", itemprop="name") else None,
                "Author": book.find("meta", itemprop="author")["content"] if book.find("meta", itemprop="author") else None,
                "Year": book.find("meta", itemprop="datePublished")["content"] if book.find("meta", itemprop="datePublished") else None,
                "Price": price_tag["content"] if price_tag else None,
                "Currency": currency_meta["content"] if currency_meta else None,
                "Seller Rating": book.find("span", class_="bookseller-rating").get_text(strip=True) if book.find("span", class_="bookseller-rating") else None
            }

            # appending the book dictionary to the list of books data
            books_data.append(book_dict)
        
        # incrementing the page number and item count for the next iteration of the loop, needed for this website specifically
        page_number += 1
        items += 20

    # handling any exceptions that occur during the scraping process and printing an error message
    except Exception as e:
        print("An error occurred: ", e)
        break

# creating a DataFrame from the list of books data and saving it to a CSV file
df = pd.DataFrame(books_data)

# saving the DataFrame to a CSV file named 'techreads_books.csv', and printing a confirmation message
df.to_csv('techreads_books.csv')
print("Data saved to techreads_books.csv")

page:  0
https://www.abebooks.co.uk/servlet/SearchResults?ds=20&dym=on&kn=data%20engineering&p=0&rollup=on&sortby=20&sp=0&spo=0
page:  1
https://www.abebooks.co.uk/servlet/SearchResults?ds=20&dym=on&kn=data%20engineering&p=1&rollup=on&sortby=20&sp=0&spo=20
Data saved to techreads_books.csv
